# 8장 무선 전송과 실기 과제

이 노트북은 웹 교재 [8장 무선 전송과 실기 과제](https://pidocs.aifusionlab.org/ai-python/ch08/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 8.1 Upload 모드 프로그램 구조

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch08/8-1/)

In [ ]:
import event, time, cyberpi

count = 0

@event.start
def on_start():
    cyberpi.console.println("A: +1  B: reset")

@event.is_press('a')
def on_a():
    global count
    count += 1
    cyberpi.console.println(count)

@event.is_press('b')
def on_b():
    global count
    count = 0
    cyberpi.console.println("reset")

## 8.2 Wi-Fi 접속과 측정값 전송

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch08/8-2/)

In [ ]:
NODE_ID = "S01"

def make_body(zone, rssi, ssid):
    return ('{"node_id":"%s","zone":"%s","rssi":%d,"ssid":"%s"}'
            % (NODE_ID, zone, rssi, ssid))

print(make_body("D1", -55, "ICT"))

In [ ]:
import socket

SERVER_IP = "교수자가 안내한 서버 주소"
UDP_PORT = 8611

def send_udp(body):
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.settimeout(3)                            # 응답을 3초까지 기다림
    try:
        s.sendto(body.encode(), (SERVER_IP, UDP_PORT))
        resp = s.recv(64)
        return b"200" in resp                  # 서버가 정상 수신하면 True
    except Exception:
        return False
    finally:
        s.close()

def send_measure(wlan, body):
    if not wlan.isconnected():                 # 끊겨 있으면 재접속
        wlan.connect(WIFI_SSID, WIFI_PASS)
        time.sleep(5)
    ok = send_udp(body)
    if not ok:                                 # 실패하면 1회 재시도
        ok = send_udp(body)
    cyberpi.console.println("send OK" if ok else "send FAIL")
    return ok

## 8.5 제스처 데이터 수집

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch08/8-5/)

In [ ]:
import pandas as pd

g = pd.read_csv("g_shake_07.csv")
print(g["rep"].nunique(), "회")
print(g.groupby("rep").size().describe().round(1))
g.groupby("rep")[["ax", "ay", "az"]].std().mean().round(2)